# MODELO ESTIMACIÓN PROBABILIDAD DE REVERSIÓN FUTUROS CRIPTOMONEDAS - BINANCE

In [1]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
import pandas as pd
import numpy as np
import ta
import joblib

import time
from datetime import datetime, timezone, timedelta

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV

import warnings
warnings.filterwarnings("ignore")

# 1. ENTRENAMIENTO DEL MODELO

## 🎯 Objetivo del modelo

Detectar reversiones de sobrecompra / sobreventa en 1H, con resultado en ≤ 4 horas.

Reversiones fuertes en ≤ 4 horas en timeframe 1h, con confirmación posterior en 15m.

👉 Esto implica que NO necesitas años de datos, necesitas suficientes ciclos completos de mercado.

Parte	Función
build_dataset	Construye datos históricos
label_reversal	Define qué fue una reversión
train_model	Entrena el modelo
scan_market_ml	Usa el modelo entrenado
predict_proba	Devuelve probabilidad


In [2]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BASE_URL = "https://fapi.binance.com"
BINANCE_FUTURES_BASE = "https://fapi.binance.com"

#INTERVAL = "1h"
INTERVAL = "15m"
TRAIN_SIZE = 2000                  # Número de velas a descargar para entrenar el modelo (cantidad de datapoints)
SCAN_SIZE = 120
horizon = 4                  # horas futuras
#REV_PCT = 0.04               # 4% reversión

MAX_SYMBOLS_TRAIN = 50       # limitar universo de entrenamiento
GMT5 = timezone(timedelta(hours=-5))
GMT = timezone(timedelta(hours=0))

# =========================================================
# 1️⃣ UNIVERSO BINANCE FUTURES
# =========================================================
def get_all_usdt_futures_symbols():
    """
    Retorna todos los símbolos de futuros perpetuos USDT en Binance.
    """
    response = requests.get(
        f"{BASE_URL}/fapi/v1/exchangeInfo",
        timeout=10
    )
    response.raise_for_status()

    data = response.json()

    return [
        s["symbol"] for s in data["symbols"]
        if s["contractType"] == "PERPETUAL"
        and s["quoteAsset"] == "USDT"
        and s["status"] == "TRADING"
    ]


# =========================================================
# 1️⃣.1 FILTRANDO UNIVERSO BINANCE FUTURES PARA OBTENER LOS SÍMBOLOS ESTABLES, LÍQUIDOS Y REPRESENTATIVOS PARA ENTRENAMIENTO DEL MODELO
# =========================================================
def select_training_symbols(
    max_symbols,
    min_volume_usdt=100_000_000
):
    """
    Selecciona símbolos USDT Futures líquidos y estables
    para entrenamiento del modelo de reversión.
    """

    symbols = get_all_usdt_futures_symbols()
    tickers = requests.get(
        f"{BASE_URL}/fapi/v1/ticker/24hr",
        timeout=10
    ).json()

    rows = []

    for t in tickers:
        sym = t["symbol"]
        if sym not in symbols:
            continue

        volume = float(t["quoteVolume"])
        if volume < min_volume_usdt:
            continue

        rows.append({
            "symbol": sym,
            "volume": volume,
            "price_change": abs(float(t["priceChangePercent"]))
        })

    df = pd.DataFrame(rows)

    # Priorizar liquidez + movimiento razonable
    df = df.sort_values(
        ["volume", "price_change"],
        ascending=[False, True]
    )

    return df["symbol"].head(max_symbols).tolist()



# =========================================================
# 2️⃣ OHLC
# =========================================================

def get_ohlc(
    symbol: str,
    interval: str,
    total_limit: int
) -> pd.DataFrame:
    """
    Descarga OHLC histórico fragmentando llamadas
    para superar el límite de Binance (1500).
    """

    MAX_LIMIT = 1500
    all_dfs = []
    remaining = total_limit
    end_time = None

    while remaining > 0:
        fetch_limit = min(MAX_LIMIT, remaining)

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch_limit
        }

        if end_time:
            params["endTime"] = end_time

        r = requests.get(
            f"{BINANCE_FUTURES_BASE}/fapi/v1/klines",
            params=params,
            timeout=10
        )
        r.raise_for_status()
        data = r.json()

        if not data:
            break

        df = pd.DataFrame(data, columns=[
            "open_time","open","high","low","close","volume",
            "close_time","qav","trades","tb_base","tb_quote","ignore"
        ])

        df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
        for c in ["open","high","low","close","volume"]:
            df[c] = df[c].astype(float)

        df = df[["time","open","high","low","close","volume"]]

        all_dfs.append(df)

        # preparar siguiente fragmento (retroceder)
        end_time = int(df["time"].iloc[0].timestamp() * 1000) - 1
        remaining -= fetch_limit

        time.sleep(0.15)  # rate-limit safety

    if not all_dfs:
        return pd.DataFrame()

    df_all = (
        pd.concat(all_dfs)
        .drop_duplicates(subset="time")
        .sort_values("time")
        .reset_index(drop=True)
    )

    return df_all





# =========================================================
# 3️⃣ INDICADORES TÉCNICOS (SIMÉTRICOS LONG/SHORT)
# =========================================================

def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula indicadores técnicos para reversión y timing.
    Válida para scan_market_ml (1h) y confirm_entry_15m (15m).
    """

    # Momentum
    df["rsi"] = ta.momentum.rsi(df["close"], window=14)
    df["mfi"] = ta.volume.money_flow_index(
        df["high"], df["low"], df["close"], df["volume"], window=14
    )

    # Stochastic RSI
    stoch = ta.momentum.StochRSIIndicator(df["close"], window=14)
    df["stoch_k"] = stoch.stochrsi_k()
    df["stoch_d"] = stoch.stochrsi_d()

    # MACD Histogram
    macd = ta.trend.MACD(df["close"])
    df["macd_hist"] = macd.macd_diff()

    # Trend context
    df["ema50"] = ta.trend.ema_indicator(df["close"], window=50)
    df["dist_ema50"] = (df["close"] - df["ema50"]) / df["close"]

    # Volatility
    df["atr"] = ta.volatility.average_true_range(
        df["high"], df["low"], df["close"], window=14
    )
    #df["atr_rel"] = df["atr"] / df["close"]
    df["atr_rel"] = df["atr"] / df["atr"].rolling(50).mean()


    # Candle structure
    df["wick_ratio"] = (df["high"] - df["close"]) / (
        (df["high"] - df["low"]) + 1e-9
    )

    # Volume confirmation
    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean()

    # Limpieza mínima
    df = df.dropna().reset_index(drop=True)

    return df


# =========================================================
# 4️⃣ LABELS (REVERSIÓN REAL LONG + SHORT)
# =========================================================
def label_reversal(
    df,
    horizon=horizon,
    atr_mult=1.2
):
    """
    Label = 1 si el precio se expande ≥ atr_mult * ATR
    dentro de las próximas `horizon` velas (1h).
    """

    df = df.copy()

    future_high = (
        df["high"]
        .shift(-1)
        .rolling(horizon)
        .max()
    )

    future_low = (
        df["low"]
        .shift(-1)
        .rolling(horizon)
        .min()
    )

    # LONG: expansión alcista
    long_hit = (
        future_high - df["close"]
    ) >= (df["atr"] * atr_mult)

    # SHORT: expansión bajista
    short_hit = (
        df["close"] - future_low
    ) >= (df["atr"] * atr_mult)

    df["label"] = (long_hit | short_hit).astype(int)

    return df.dropna()



# =========================================================
# 5️⃣ DATASET SUPERVISADO
# =========================================================
def build_dataset(
    symbols,
    interval="1h",
    horizon=horizon,
    limit=TRAIN_SIZE,
    max_symbols=MAX_SYMBOLS_TRAIN,
    atr_mult=1.0   # 🔽 más realista para 1h
):
    rows = []

    for sym in symbols[:max_symbols]:
        try:
            df = get_ohlc(sym, interval=interval, total_limit=limit)
            df = add_indicators(df)
            
            if df is None or df.empty or len(df) < 100:
                continue

            df = label_reversal(
                df,
                horizon=horizon,
                atr_mult=atr_mult
            )

            for i in range(len(df) - horizon):
                row = df.iloc[i]

                # 🔧 Filtro de edge RELAJADO (clave)
                if not (
                    row.rsi < 40 or
                    row.rsi > 60 or
                    abs(row.dist_ema50) > 0.015
                ):
                    continue

                rows.append({
                    **{f: float(row[f]) for f in FEATURES},
                    "label": int(row.label),
                    "Symbol": sym,
                    "Direction": "LONG" if row.rsi < 50 else "SHORT",
                    "Entry_Price": float(row.close),
                    "ATR_rel": float(row.atr_rel),
                    "Timestamp_UTC": pd.to_datetime(row.time, utc=True)
                })

        except Exception as e:
            print(f"[WARN build_dataset] {sym}: {e}")
            continue

    if not rows:
        raise RuntimeError(
            "build_dataset produjo 0 filas. "
            "Relaja filtros o reduce atr_mult."
        )

    df_out = pd.DataFrame(rows).dropna().reset_index(drop=True)

    print(f"[INFO] Dataset construido: {len(df_out)} filas")

    return df_out



# =========================================================
# 6️⃣ MODELO ML (EXPLICABLE)
# =========================================================

def train_model(dataset, split_ratio=0.8):
    dataset = dataset.sort_values("Timestamp_UTC")

    split = int(len(dataset) * split_ratio)
    train = dataset.iloc[:split]
    val   = dataset.iloc[split:]

    X_train = train[FEATURES]
    y_train = train["label"]

    X_val = val[FEATURES]
    y_val = val["label"]

    base_model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            class_weight="balanced",
            max_iter=600,
            C=0.8,
            solver="lbfgs"
        ))
    ])

    # 🔹 Calibración de probabilidades
    model = CalibratedClassifierCV(
        base_model,
        method="isotonic",
        cv=3
    )

    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, prob_val)

    print(f"ROC-AUC validación temporal: {round(auc,3)}")
    print(f"Samples train: {len(train)} | val: {len(val)}")

    return model





# =========================================================
# 7️⃣ SCANNER FINAL (LONG + SHORT)
# =========================================================

# ==============================
# CONFIG
# ==============================
GMT5 = timezone(timedelta(hours=-5))

FEATURES = [
    # 🔴 EXTREMOS
    "rsi",          # sobrecompra / sobreventa
    "mfi",          # flujo real de dinero
    "stoch_k",      # timing
    "stoch_d",

    # 🟡 MOMENTUM
    "macd_hist",    #
    
    # 🔵 CONTEXTO DE PRECIO
    "dist_ema50",   # qué tan estirado está el precio

    # 🟣 VOLATILIDAD / RIESGO
    "atr_rel",      # ATR normalizado
    
    # 🟠 MICROESTRUCTURA
    "wick_ratio",   # rechazo de precio
    "vol_ratio"     # expansión o absorción
]


# ==============================
# SCANNER DEFINITIVO
# ==============================
def scan_market_ml(
    model,
    min_prob=0.75,
    interval="1h",
    limit=SCAN_SIZE
):
    """
    Escanea todos los futuros USDT de Binance y retorna
    los símbolos con mayor probabilidad de reversión.
    """

    symbols = get_all_usdt_futures_symbols()
    results = []
    scan_time = datetime.now(GMT5)

    for symbol in symbols:
        try:
            # 1️⃣ Descargar datos
            df = get_ohlc(symbol, interval=interval, total_limit=limit)
            df = add_indicators(df)

            if df is None or df.empty or len(df) < 50:
                continue

            last = df.iloc[-1]

            # 2️⃣ Validar indicadores
            if any(pd.isna(last[f]) for f in FEATURES):
                continue

            # 3️⃣ Precio base de la señal (ANCLA)
            entry_price = float(last["close"])

            # 4️⃣ Preparar features para ML
            X = pd.DataFrame(
                [[last[f] for f in FEATURES]],
                columns=FEATURES
            )

            # 5️⃣ Inferencia ML (ROBUSTA)
            proba = model.predict_proba(X)

            if proba.shape[1] > 1:
                prob_reversal = float(proba[0][1])
            else:
                prob_reversal = float(proba[0][0])

            if prob_reversal < min_prob:
                continue

            # 6️⃣ Inferir dirección (NO ML)
            direction, dir_conf = infer_reversal_direction(last)

            # 7️⃣ Guardar resultado
            results.append({
                "Symbol": symbol,
                "Direction": direction,
                "Prob_Reversal": round(prob_reversal, 4),
                "Dir_Confidence": round(dir_conf, 3),
                "Entry_Price": round(entry_price, 6),
                "ATR": round(float(last["atr"]), 6),
                "ATR_rel": round(float(last["atr_rel"]), 4),

                # Indicadores (transparencia / auditoría)
                "RSI": round(float(last["rsi"]), 2),
                "MFI": round(float(last["mfi"]), 2),
                "Stoch_K": round(float(last["stoch_k"]), 2),
                "Stoch_D": round(float(last["stoch_d"]), 2),
                "MACD_hist": round(float(last["macd_hist"]), 6),

                # Timestamps CORRECTOS
                "Signal_Time": last["time"],          # vela base
                "Scan_Time_GMT5": scan_time            # momento del escaneo
            })

        except Exception as e:
            print(f"[WARN] {symbol}: {e}")
            continue

    if not results:
        return pd.DataFrame()

    df_out = pd.DataFrame(results)

    # 8️⃣ Validación defensiva final
    for col in ["Prob_Reversal", "Dir_Confidence"]:
        if col not in df_out.columns:
            raise RuntimeError(f"scan_market_ml inconsistente: falta {col}")

    # 9️⃣ Ranking final
    df_out = df_out.sort_values(
        ["Prob_Reversal", "Dir_Confidence"],
        ascending=False
    ).reset_index(drop=True)

    return df_out



# =========================================================
#  FUNCIÓN DE DIRECCIÓN DE REVERSIÓN
# =========================================================
def infer_reversal_direction(row):
    """
    Determina la dirección más probable de la reversión
    usando reglas técnicas interpretables.
    """

    long_score = 0.0
    short_score = 0.0

    # ---------- MOMENTUM ----------
    if row["rsi"] < 35:
        long_score += 1.5
    if row["rsi"] > 65:
        short_score += 1.5

    if row["mfi"] < 35:
        long_score += 1.5
    if row["mfi"] > 65:
        short_score += 1.5

    # ---------- STOCH RSI ----------
    if row["stoch_k"] < 20 and row["stoch_d"] < 20:
        long_score += 1.0
    if row["stoch_k"] > 80 and row["stoch_d"] > 80:
        short_score += 1.0

    # ---------- EXTENSIÓN ----------
    if row["dist_ema50"] < -0.025:
        long_score += 1.5
    if row["dist_ema50"] > 0.025:
        short_score += 1.5

    # ---------- MOMENTUM SHIFT (MACD HIST) ----------
    if row["macd_hist"] > 0:
        long_score += 0.75
    if row["macd_hist"] < 0:
        short_score += 0.75

    # ---------- ESTRUCTURA DE VELA ----------
    if row["wick_ratio"] > 0.6:
        long_score += 0.75
    if row["wick_ratio"] < 0.4:
        short_score += 0.75

    # ---------- VOLUMEN ----------
    if row["vol_ratio"] > 1.2:
        long_score += 0.5
        short_score += 0.5  # confirma interés, no dirección

    # ---------- DECISIÓN ----------
    total = long_score + short_score + 1e-6

    if long_score >= short_score:
        return "LONG", round(long_score / total, 3)
    else:
        return "SHORT", round(short_score / total, 3)



## 6️⃣ Importante: estabilidad temporal

⚠️ NO cambiar símbolos cada día

Reglas sanas:
Acción	Frecuencia

* Seleccionar símbolos	Cada 1–2 meses

* Entrenar modelo	Cada 1–3 semanas

* Inferencia	Tiempo real

📌 Si cambias símbolos a menudo:

* rompes coherencia estadística

* destruyes calibración de probabilidad

In [3]:
# =========================================================
# 8️⃣ EJECUCIÓN
# =========================================================
if __name__ == "__main__":
    print("📦 Filtrando símbolos estables…")
    symbols = select_training_symbols(max_symbols=MAX_SYMBOLS_TRAIN)
    
    print("📦 Construyendo dataset…")
    dataset = build_dataset(symbols, limit=TRAIN_SIZE, horizon=horizon)

    print("🧠 Entrenando modelo…")
    model = train_model(dataset)

    print("🔍 Escaneando mercado…")
    signals = scan_market_ml(model, min_prob=0.75)

📦 Filtrando símbolos estables…
📦 Construyendo dataset…
[WARN build_dataset] SPACEUSDT: index 13 is out of bounds for axis 0 with size 7
[INFO] Dataset construido: 37921 filas
🧠 Entrenando modelo…
ROC-AUC validación temporal: 0.811
Samples train: 30336 | val: 7585
🔍 Escaneando mercado…
[WARN] SPACEUSDT: index 13 is out of bounds for axis 0 with size 7
[WARN] FIGHTUSDT: index 13 is out of bounds for axis 0 with size 6


In [4]:
len(symbols)

39

In [5]:
signals

,Symbol,Direction,Prob_Reversal,Dir_Confidence,Entry_Price,ATR,ATR_rel,RSI,MFI,Stoch_K,Stoch_D,MACD_hist,Signal_Time,Scan_Time_GMT5
0,JELLYJELLYUSDT,LONG,1.0000,0.857,0.060070,0.000839,1.0728,44.92,43.49,0.59,0.78,0.000120,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
1,TURTLEUSDT,LONG,1.0000,0.818,0.055630,0.000877,1.0913,62.47,64.87,0.71,0.44,0.000090,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
2,COMMONUSDT,SHORT,1.0000,0.690,0.002747,0.000060,1.2608,73.75,67.84,0.88,0.78,0.000014,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
3,EIGENUSDT,SHORT,1.0000,0.654,0.347300,0.005789,0.9970,64.60,66.45,0.88,0.60,0.001010,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
4,BLUAIUSDT,SHORT,1.0000,0.625,0.007460,0.000769,4.7116,77.49,99.31,0.78,0.53,0.000093,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383,CHZUSDT,LONG,0.7579,0.700,0.051050,0.000918,0.9238,51.07,43.55,0.40,0.24,0.000007,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
384,RESOLVUSDT,LONG,0.7567,0.700,0.100690,0.002827,0.8699,47.03,46.96,0.23,0.22,-0.000311,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
385,ZROUSDT,LONG,0.7567,0.625,2.303900,0.070972,1.1770,61.62,58.82,0.53,0.51,0.000363,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00
386,LIGHTUSDT,SHORT,0.7567,0.600,0.444700,0.008145,0.9252,50.84,41.72,0.50,0.32,-0.000641,2026-01-23 17:00:00+00:00,2026-01-23 12:43:53.238715-05:00


## Guardar Modelo en Disco Duro

In [68]:
current_date = str(datetime.now())[:16].replace(":", "").replace(" ", "_")
model_filename = 'Modelo_reversion_futuros_cripto_' + str(horizon) + "H_" + current_date + '.joblib'
joblib.dump(model, model_filename)
print(f"Model saved successfully as {model_filename}")

Model saved successfully as Modelo_reversion_futuros_cripto_4H_2026-01-21_1956.joblib


## Guardar resultados del scanner en CSV en memoria

In [69]:
def save_scanner_results_to_disk(
    signals: pd.DataFrame,
    output_dir: str,
    INTERVAL: str,
    horizon: int
):
    """
    Guarda los resultados del scanner en un CSV en disco.
    El nombre del archivo incluye la estampa de tiempo de la última vela analizada.

    Returns:
        full_path (str): ruta completa del archivo guardado
    """
    if signals.empty:
        raise ValueError("El dataframe de señales está vacío")

    # Asegurar directorio
    os.makedirs(output_dir, exist_ok=True)

    # Asegurar datetime
    signals["Signal_Time"] = pd.to_datetime(signals["Signal_Time"], utc=True)

    # Guardar parámetros de diseño del modelo
    signals["Timeframe"] = INTERVAL
    signals["Horizon_Eval"] = str(horizon)+"h"

    # Última vela utilizada
    last_candle_time = signals["Signal_Time"].max()
    timestamp_str = last_candle_time.strftime("%Y%m%d_%H%MUTC")

    filename = f"scanner_results_{timestamp_str}.csv"
    full_path = os.path.join(output_dir, filename)

    signals.to_csv(full_path, index=False)

    return full_path


In [70]:
output_dir = "C:/Users/Lenovo S540/Python_scripts"
full_path = save_scanner_results_to_disk(signals, output_dir, INTERVAL, horizon)

print(f"Archivo guardado en la ruta {full_path}")


Archivo guardado en la ruta C:/Users/Lenovo S540/Python_scripts\scanner_results_20260122_0000UTC.csv


# 2. EVALUACIÓN DEL MODELO

## Cargar resultados del scanner desde disco

In [ ]:
def load_scanner_results_from_disk(
    output_dir: str,
    filename: str
):
    """
    Carga un archivo CSV específico desde disco a un DataFrame.
    """
    full_path = os.path.join(output_dir, filename)

    if not os.path.exists(full_path):
        raise FileNotFoundError(f"No existe el archivo: {full_path}")

    df = pd.read_csv(full_path)

    # Restaurar datetime
    if "Signal_Time" in df.columns:
        df["Signal_Time"] = pd.to_datetime(df["Signal_Time"], utc=True)

    return df


In [ ]:
signals = load_scanner_results_from_disk(
    output_dir="C:/Users/Lenovo S540/Python_scripts",
    filename="scanner_results_20251223_1828UTC.csv"
)

signals.head()


## Evaluar expansión real post-señal (sin backtest clásico)

In [ ]:
def evaluate_post_signal_expansion(
    df,
    signal_time,
    entry_price,
    direction,
    atr_at_signal,
    horizon=4,
    expansion_threshold=1.0
):
    """
    Evalúa la expansión real post-señal usando datos ya descargados.
    """

    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    signal_time = pd.to_datetime(signal_time, utc=True)

    future = df[df["time"] > signal_time].head(horizon)

    if len(future) < horizon:
        return None

    if direction == "LONG":
        mfe = (future["high"].max() - entry_price) / atr_at_signal
        mae = (entry_price - future["low"].min()) / atr_at_signal
    else:
        mfe = (entry_price - future["low"].min()) / atr_at_signal
        mae = (future["high"].max() - entry_price) / atr_at_signal

    return {
        "MFE_R": round(mfe, 2),
        "MAE_R": round(mae, 2),
        "Expanded": mfe >= expansion_threshold
    }


In [ ]:
for _, row in signals.iterrows():
    df_price = price_data[row.Symbol]  # ya descargado
    result = evaluate_post_signal_expansion(
        df=df_price,
        signal_time=row.Signal_Time,
        entry_price=row.Entry_Price,
        direction=row.Direction,
        atr_at_signal=row.ATR_at_Signal
    )


In [ ]:
expansion_df

## Medir MAE previo a expansión
Experimentar con diferentes cantidades de velas de 15 minutos a esperar para hacer la entrada mediante la variable max_wait

In [ ]:
def mae_before_expansion(
    symbol,
    signal_time,
    direction,
    entry_price,
    atr,
    max_wait=4
):
    df = get_ohlc(symbol, interval="15m", total_limit=max_wait + 2)
    df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)

    future = df[df["time"] > signal_time].head(max_wait)

    if direction == "LONG":
        mae = (future["low"].min() - entry_price) / atr
    else:
        mae = (entry_price - future["high"].max()) / atr

    return round(mae, 2)


In [ ]:
symbol = "LIGHTUSDT"

# entry_price = 0.301254
entry_price = signals["Entry_Price"][signals["Symbol"]==symbol].item()
atr = signals["ATR_rel"][signals["Symbol"]==symbol].item()
direction = signals["Direction"][signals["Symbol"]==symbol].item()

# signal_time = pd.Timestamp("2025-01-15 14:00:00", tz="UTC")
signal_time = signals["Signal_Time"][signals["Symbol"]==symbol].item()

mae_before_expansion(symbol, signal_time, direction, entry_price, atr, max_wait=4)

## Reporte de Calidad de Expansión

In [ ]:
def expansion_quality_report(df):
    return {
        "Signals": len(df),
        "Expansion_Rate_%": round(df["Expanded"].mean()*100, 1),
        "Avg_MFE_ATR": round(df["MFE_ATR"].mean(), 2),
        "Avg_MAE_ATR": round(df["MAE_ATR"].mean(), 2),
        "Edge_Ratio": round(
            df["MFE_ATR"].mean() / abs(df["MAE_ATR"].mean()), 2
        )
    }

## Comparar Horizontes

In [ ]:
def compare_horizons(expansion_df):
    return (
        expansion_df
        .groupby("Horizon")
        .agg(
            Signals=("Expanded", "count"),
            Expansion_Rate=("Expanded", "mean"),
            Avg_MFE_ATR=("MFE_ATR", "mean"),
            Avg_MAE_ATR=("MAE_ATR", "mean")
        )
        .assign(
            Expansion_Rate_%=lambda x: (x["Expansion_Rate"]*100).round(1),
            Edge_Ratio=lambda x: (x["Avg_MFE_ATR"] / x["Avg_MAE_ATR"].abs()).round(2)
        )
        .reset_index()
    )


In [ ]:
expansion_df = compare_horizons(expansion_df)
expansion_df

# 3. AGREGACIÓN Y RETROALIMENTACIÓN

## Buckets de probabilidad

In [ ]:
def expansion_by_probability_bucket(df):
    df = df.copy()

    df["Prob_Bucket"] = pd.cut(
        df["Prob_Reversal"],
        bins=[0.7, 0.75, 0.8, 0.85, 0.9, 1.0],
        right=False
    )

    summary = (
        df.groupby("Prob_Bucket")
          .agg(
              Signals=("Expanded", "count"),
              Expansion_Rate=("Expanded", "mean"),
              Avg_MFE_ATR=("MFE_ATR", "mean"),
              Avg_MAE_ATR=("MAE_ATR", "mean")
          )
          .reset_index()
    )

    summary["Expansion_Rate_%"] = (summary["Expansion_Rate"] * 100).round(1)
    summary["Edge_Ratio"] = (
        summary["Avg_MFE_ATR"] / summary["Avg_MAE_ATR"].abs()
    ).round(2)

    return summary.drop(columns=["Expansion_Rate"])

def label_expansion(df, horizon=HORIZON, atr_mult=1.2):
    future_high = df["high"].shift(-1).rolling(horizon).max()
    future_low  = df["low"].shift(-1).rolling(horizon).min()

    long_expansion  = (future_high - df["close"]) >= atr_mult * df["atr"]
    short_expansion = (df["close"] - future_low) >= atr_mult * df["atr"]

    df["label"] = (long_expansion | short_expansion).astype(int)
    return df.dropna()



In [ ]:
expansion_df = expansion_by_probability_bucket(expansion_df)
expansion_df

# 4. EJECUCIÓN DEL MODELO (INFERENCIA)

## Flujo operativo final

1️⃣ Scanner 1H detecta reversión
2️⃣ Bucket → riesgo dinámico
3️⃣ Calcular SL / TP por ATR
4️⃣ Esperar confirmación 15M
5️⃣ Si confirma → entrar
6️⃣ Si no confirma → descartar
7️⃣ Gestión pasiva hasta TP / SL



### ⏱️ VENTANA DE ESPERA (CLAVE)

Máximo: 3–4 velas de 15m

Si no confirma → cancelar trade

Esto evita: entradas tardías y operar señales “muertas”


### CONDICIONES DE ENTRADA 15M (REVERSIONES)



#### 📈 Para LONG (rebote):

Entrar SOLO si ocurre al menos 2 de 3:

• RSI 15m cruza ↑ 30

• MACD histogram pasa de ↓ a ↑

• Vela de rechazo (pin bar / engulfing)



#### 📉 Para SHORT (caída):

Entrar SOLO si ocurre al menos 2 de 3:

• RSI 15m cruza ↓ 70

• MACD histogram pasa de ↑ a ↓

• Vela de rechazo superior

## Cargar Modelo anterior

In [3]:
model_filename = "Modelo_reversion_futuros_cripto_4H_2026-01-21_1956.joblib"

model = joblib.load(model_filename)
print(f"Modelo {model_filename} cargado exitosamente")

Modelo Modelo_reversion_futuros_cripto_4H_2026-01-21_1956.joblib cargado exitosamente


## Generar Inferencias

¿Hacia dónde es más probable la reversión? → Dir_Confidence: Claridad direccional de la reversión, confianza relativa (0-1)


¿Cuánto puede moverse en términos absolutos? → ATR: Cuánto se mueve este activo en promedio por vela de 1h en USDT                                                                                                  

< 0.55	Señal ambigua → NO operar

0.55 – 0.65	Reversión posible, timing crítico

0.65 – 0.75	Reversión clara

'> 0.75	Reversión muy bien alineada

¿Qué tan explotable es ese movimiento en términos relativos? → ATR_rel: Qué % del precio puede moverse por hora

In [4]:
print("🔍 Escaneando mercado…")
signals = scan_market_ml(model, min_prob=0.75)
signals

🔍 Escaneando mercado…
[WARN] ELSAUSDT: index 13 is out of bounds for axis 0 with size 8
[WARN] SKRUSDT: index 13 is out of bounds for axis 0 with size 5


,Symbol,Direction,Prob_Reversal,Dir_Confidence,Entry_Price,ATR,ATR_rel,RSI,MFI,Stoch_K,Stoch_D,MACD_hist,Signal_Time,Scan_Time_GMT5
0,0GUSDT,SHORT,0.9968,0.719,0.831700,0.018790,1.4167,75.07,90.36,0.90,0.64,0.006449,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
1,SPORTFUNUSDT,SHORT,0.9968,0.719,0.079670,0.003252,1.1521,73.83,83.86,1.00,0.85,0.001441,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
2,CUSDT,LONG,0.9937,0.808,0.069290,0.000709,0.8099,33.66,36.07,0.02,0.04,-0.000157,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
3,JCTUSDT,SHORT,0.9937,0.719,0.001844,0.000038,0.8990,70.44,81.01,0.91,0.72,0.000008,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
4,CHESSUSDT,LONG,0.9937,0.706,0.028190,0.000328,0.8949,45.86,28.55,0.06,0.17,-0.000026,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
5,SENTUSDT,SHORT,0.9807,0.550,0.024200,0.002228,2.7951,63.51,53.65,0.74,0.72,0.000462,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
6,RECALLUSDT,LONG,0.9807,0.529,0.085900,0.001329,1.0039,61.01,75.73,0.88,0.55,0.000162,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
7,BRETTUSDT,LONG,0.9710,0.750,0.013840,0.000215,0.8419,43.63,18.26,0.00,0.07,-0.000028,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
8,CUDISUSDT,LONG,0.9710,0.643,0.028800,0.000381,0.7918,43.80,37.48,0.55,0.32,0.000034,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00
9,2ZUSDT,LONG,0.9710,0.643,0.124500,0.002393,1.0652,48.23,53.67,0.37,0.40,-0.000232,2026-01-22 14:00:00+00:00,2026-01-22 09:21:19.563263-05:00


## Confirmación de entrada 15m

In [8]:
TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")


In [9]:
def get_15m_requirements(prob_reversal):
    if prob_reversal >= 0.90:
        return 1.5
    elif prob_reversal >= 0.80:
        return 2.0
    else:
        return 2.5


In [10]:
def add_indicators_15m_sequence(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula indicadores técnicos para CONFIRMACIÓN DE ENTRADA (15m).
    Optimizada para análisis secuencial y tiempo real.
    
    ❗ NO usar para scanner 1H
    ❗ NO usar para entrenamiento ML
    """

    df = df.copy()

    # ================= MOMENTUM =================
    # RSI clásico: se usa la PENDIENTE, no el nivel absoluto
    df["rsi"] = ta.momentum.rsi(
        df["close"],
        window=14
    )

    # ================= MACD =================
    # Usamos HISTOGRAMA (macd_diff) para detectar giro de momentum
    macd = ta.trend.MACD(
        df["close"],
        window_fast=12,
        window_slow=26,
        window_sign=9
    )
    df["macd_hist"] = macd.macd_diff()

    # ================= VOLATILIDAD =================
    # ATR rápido para:
    # - dimensionar SL / TP
    # - filtrar ruido extremo
    df["atr"] = ta.volatility.average_true_range(
        df["high"],
        df["low"],
        df["close"],
        window=14
    )

    # ================= ESTRUCTURA DE VELA =================
    # Wick ratio:
    # LONG  -> mechas inferiores largas
    # SHORT -> mechas superiores largas
    df["wick_ratio"] = (df["high"] - df["close"]) / (
        (df["high"] - df["low"]) + 1e-9
    )

    # ================= VOLUMEN =================
    # Confirmación local (no régimen)
    df["vol_ratio"] = (
        df["volume"] /
        df["volume"].rolling(20).mean()
    )

    # ================= LIMPIEZA =================
    # Mantener secuencia completa (NO solo última vela)
    df = df.dropna().reset_index(drop=True)

    return df


In [11]:
def confirm_entry_15m_realtime(
    symbol: str,
    direction: str,
    prob_reversal: float,
    max_wait_minutes: int = 60,
    poll_seconds: int = 60
):
    """
    Monitorea velas de 15m en tiempo real y envía
    alerta Telegram cuando se cumplan condiciones de entrada.
    """
    req = get_15m_requirements(prob_reversal)
    confirmations = 0.0
    
    start_time = datetime.now(timezone.utc)
    end_time = start_time + timedelta(minutes=max_wait_minutes)

    last_checked_time = None

    send_telegram_alert(
        f"🕵️ Iniciando monitoreo 15m para {symbol} ({direction})\n"
        f"⏱️ Tiempo máximo: {max_wait_minutes} minutos"
    )

    while (datetime.now(timezone.utc) < end_time) and (confirmations < req):

        try:
            df15 = get_ohlc(symbol, interval="15m", total_limit=80)
            df15 = add_indicators_15m_sequence(df15)
            df15_valid = df15.dropna()
            
            print("LEN df15_valid:", len(df15_valid))
            if len(df15_valid) < 3:
                time.sleep(poll_seconds)
                continue

            last = df15_valid.iloc[-1]
            prev = df15_valid.iloc[-2]

            # Evitar reevaluar la misma vela
            if last_checked_time == last.time:
                time.sleep(poll_seconds)
                continue

            last_checked_time = last.time
            print("ENTRÓ A LOS CONDICIONALES DE DIRECCION")
            confirmations = 0.0
            reasons = []

            # ================= LONG =================
            if direction == "LONG":

                if prev.rsi < 40 and last.rsi > prev.rsi:
                    confirmations += 1
                    reasons.append("RSI turning up")
                    print(f"{datetime.now()} - {symbol}: RSI turning up")

                if prev.macd_hist < last.macd_hist:
                    confirmations += 1
                    reasons.append("MACD momentum shift")
                    print(f"{datetime.now()} - {symbol}: MACD momentum shift")

                if last.close > last.open and last.wick_ratio > 0.55:
                    confirmations += 1
                    reasons.append("Bullish rejection candle")
                    print(f"{datetime.now()} - {symbol}: Bullish rejection candle")

                if last.vol_ratio > 1.0:
                    confirmations += 0.5
                    reasons.append("Volume confirmation")
                    print(f"{datetime.now()} - {symbol}: Volume confirmation")

            # ================= SHORT =================
            else:

                if prev.rsi > 60 and last.rsi < prev.rsi:
                    confirmations += 1
                    reasons.append("RSI turning down")
                    print(f"{datetime.now()} - {symbol}: RSI turning down")

                if prev.macd_hist > last.macd_hist:
                    confirmations += 1
                    reasons.append("MACD momentum shift")
                    print(f"{datetime.now()} - {symbol}: MACD momentum shift")

                if last.close < last.open and last.wick_ratio < 0.45:
                    confirmations += 1
                    reasons.append("Bearish rejection candle")
                    print(f"{datetime.now()} - {symbol}: Bearish rejection candle")

                if last.vol_ratio > 1.0:
                    confirmations += 0.5
                    reasons.append("Volume confirmation")
                    print(f"{datetime.now()} - {symbol}: Volume confirmation")

            print("confirmations:", confirmations)
            # ================= DECISIÓN =================
            if confirmations >= req:

                msg = (
                    f"🚨 SEÑAL DE ENTRADA CONFIRMADA 🚨\n\n"
                    f"📌 Símbolo: {symbol}\n"
                    f"📈 Dirección: {direction}\n"
                    f"🕒 Vela 15m: {last.time}\n"
                    f"💰 Precio aprox: {round(last.close, 6)}\n"
                    f"   ATR(15m): {round(last.atr, 6)}\n"
                    f"✅ Confirmaciones: {confirmations}\n"
                    f"🧠 Razones:\n- " + "\n- ".join(reasons)
                )

                send_telegram_alert(msg)
                print(msg)
                return {
                    "Decision": "ENTER",
                    "Symbol": symbol,
                    "Direction": direction,
                    "Entry_Price": float(last.close),
                    "ATR_15m": float(last.atr),
                    # "ATR_rel_15m": float(last.atr_rel),
                    "Confirmations": confirmations,
                    "Reasons": reasons,
                    "Timestamp": last.time
                }

        except Exception as e:
            send_telegram_alert(f"⚠️ Error monitoreando {symbol}: {e}")
            print(f"⚠️ Error monitoreando {symbol}: {e}")

            time.sleep(poll_seconds)

    
    if confirmations < req:
        # ⛔ Timeout
        
        msg = f"⏰ Tiempo agotado sin señal de entrada\n 📌 Símbolo: {symbol}\n Dirección: {direction}"
        
        send_telegram_alert(msg)
        print(msg)
    
        return {
            "Decision": "TIMEOUT",
            "Symbol": symbol,
            "Direction": direction
        }


In [6]:
posicion = 0

symbol = signals.loc[posicion, "Symbol"]
direction = signals.loc[posicion, "Direction"]
prob_reversal = signals.loc[posicion, "Prob_Reversal"]


#symbol = "MERLUSDT"
#direction = "LONG"
#prob_reversal = 1


print("symbol:", symbol)
print("direction:", direction)
print("prob_reversal:", prob_reversal)

symbol: JELLYJELLYUSDT
direction: LONG
prob_reversal: 1.0


In [ ]:
entry = confirm_entry_15m_realtime(
    symbol=symbol,
    direction=direction,
    prob_reversal=prob_reversal,
    max_wait_minutes=90
)

print(entry)

LEN df15_valid: 47
ENTRÓ A LOS CONDICIONALES DE DIRECCION
confirmations: 0.0
LEN df15_valid: 47


## Confirmación de salida 15m

In [ ]:
def structural_exit_conditions(direction, rsi, macd_hist, atr_rel):
    # Definir régimen
    if atr_rel < 0.005:
        regime = "low"
    elif atr_rel > 0.015:
        regime = "high"
    else:
        regime = "normal"

    if direction == "LONG":
        if regime == "low":
            return rsi > 50 or macd_hist < -0.05
        elif regime == "high":
            return rsi > 60 or macd_hist < 0.05
        else:
            return rsi > 55 or macd_hist < 0

    else:  # SHORT
        if regime == "low":
            return rsi < 50 or macd_hist > 0.05
        elif regime == "high":
            return rsi < 40 or macd_hist > -0.05
        else:
            return rsi < 45 or macd_hist > 0


In [ ]:
def monitor_trade(
    symbol: str,
    direction: str,
    entry_price: float,
    atr_15m: float,
    horizon: int,
    end_time_utc,
    poll_seconds: int = 30
):
    """
    Monitorea una orden abierta y gestiona salida dinámica
    (SL, TP parcial, BE, trailing) con alertas por Telegram.
    """

    # ================= CONFIGURACIÓN POR HORIZONTE ================= #
    if horizon >= 4:
        sl_mult = 1.2
        tp_partial_mult = 0.9
        be_mult = 1.0
        tp_mult = 2.5
        trail_mult = 1.2
    elif horizon == 2:
        sl_mult = 0.9
        tp_partial_mult = 0.7
        be_mult = 0.8
        tp_mult = 1.8
        trail_mult = 1.0
    else:
        sl_mult = 0.7
        tp_partial_mult = 0.5
        be_mult = 0.6
        tp_mult = 1.2
        trail_mult = 0.8

    # ================= ESTADO ================= #
    break_even_applied = False
    tp_partial_done = False
    invalid_signals = 0

    be_activation_candle_time = None
    last_checked_candle_time = None

    # ================= NIVELES INICIALES ================= #
    if direction == "LONG":
        sl = max(entry_price - sl_mult * atr_15m, 0.0)
        tp = entry_price + tp_mult * atr_15m
        tp_partial = entry_price + tp_partial_mult * atr_15m
    else:
        sl = entry_price + sl_mult * atr_15m
        tp = entry_price - tp_mult * atr_15m
        tp_partial = entry_price - tp_partial_mult * atr_15m

    msg = f"🟢 *TRADE OPEN*\n \
        Symbol: `{symbol}`\n \
        Direction: *{direction}*\n \
        Entry: `{round(entry_price,6)}`\n \
        SL: `{round(sl,6)}`\n \
        TP Partial: `{round(tp_partial,6)}`\n \
        TP: `{round(tp,6)}`"

    send_telegram_alert(msg)
    print(msg)

    # ================= LOOP PRINCIPAL ================= #
    while datetime.now(timezone.utc) < end_time_utc:

        try:
            df15 = get_ohlc(symbol, interval="15m", total_limit=80)
            df15 = add_indicators_15m_sequence(df15)

            if df15 is None or len(df15) < 20:
                time.sleep(poll_seconds)
                continue

            last = df15.iloc[-1]
            candle_time = last.time
            last_price = float(last.close)

            is_new_candle = candle_time != last_checked_candle_time
            last_checked_candle_time = candle_time

            # ================= STOP LOSS ================= #
            can_evaluate_sl = True
            if break_even_applied and candle_time == be_activation_candle_time:
                can_evaluate_sl = False  # ⛔ evita SL en la vela del BE

            if can_evaluate_sl:
                if direction == "LONG" and last_price <= sl:
                    msg = f"🔴 *STOP LOSS*\n{symbol} LONG\nExit: `{round(sl,6)}`"
                    send_telegram_alert(msg)
                    print(msg)
                    return {"Status": "STOP_LOSS", "Exit": sl}

                if direction == "SHORT" and last_price >= sl:
                    msg = f"🔴 *STOP LOSS*\n{symbol} SHORT\nExit: `{round(sl,6)}`"
                    send_telegram_alert(msg)
                    print(msg)
                    return {"Status": "STOP_LOSS", "Exit": sl}

            # ================= TP PARCIAL ================= #
            if not tp_partial_done:
                if (direction == "LONG" and last_price >= tp_partial) or \
                   (direction == "SHORT" and last_price <= tp_partial):

                    tp_partial_done = True
                    msg = f"📈 *TP PARCIAL*\n{symbol} {direction}\n Price: `{round(tp_partial,6)}`"
                    send_telegram_alert(msg)
                    print(msg)

            # ================= BREAK EVEN ================= #
            if not break_even_applied:
                if (direction == "LONG" and last_price >= entry_price + be_mult * atr_15m) or \
                   (direction == "SHORT" and last_price <= entry_price - be_mult * atr_15m):

                    sl = entry_price
                    break_even_applied = True
                    be_activation_candle_time = candle_time

                    msg = f"🟡 *BREAK EVEN*\n{symbol} {direction}\n SL → `{round(sl,6)}`"
                    send_telegram_alert(msg)
                    print(msg)

            # ================= TRAILING STOP ================= #
            if break_even_applied and tp_partial_done and is_new_candle:
                if direction == "LONG":
                    new_sl = max(last_price - trail_mult * atr_15m, sl)
                    sl = new_sl
                else:
                    new_sl = min(last_price + trail_mult * atr_15m, sl)
                    sl = new_sl

            # ================= TAKE PROFIT FINAL ================= #
            if (direction == "LONG" and last_price >= tp) or \
               (direction == "SHORT" and last_price <= tp):

                msg = f"🟢 *TAKE PROFIT*\n{symbol} {direction}\n Exit: `{round(tp,6)}`"
                send_telegram_alert(msg)
                print(msg)
                return {"Status": "TAKE_PROFIT", "Exit": tp}

            # ================= SALIDA ESTRUCTURAL ================= #
            
            structure_invalid = structural_exit_conditions(
                direction,
                last.rsi,
                last.macd_hist,
                atr_rel=atr_15m / last.close
            )

            invalid_signals = invalid_signals + 1 if structure_invalid else max(0, invalid_signals - 1)

            if invalid_signals >= 2:
                msg = f"⚠️ *SALIDA ESTRUCTURAL*\n{symbol} {direction}\n Price: `{round(last_price,6)}`"
                send_telegram_alert(msg)
                print(msg)
                return {"Status": "STRUCTURAL_EXIT", "Exit": last_price}

            time.sleep(poll_seconds)

        except Exception as e:
            print(f"[WARN] monitor_trade {symbol}: {e}")
            time.sleep(poll_seconds)

    # ================= TIEMPO EXPIRADO ================= #
    msg = f"⏰ *TIEMPO EXPIRADO*\n{symbol} {direction}\n Cierre manual recomendado"
    send_telegram_alert(msg)
    print(msg)
    return {"Status": "TIME_EXIT"}


In [14]:
print(entry)

{'Decision': 'ENTER', 'Symbol': 'SPORTFUNUSDT', 'Direction': 'SHORT', 'Entry_Price': 0.07508, 'ATR_15m': 0.0027016900572235793, 'Confirmations': 3.0, 'Reasons': ['RSI turning down', 'MACD momentum shift', 'Bearish rejection candle'], 'Timestamp': Timestamp('2026-01-22 15:30:00+0000', tz='UTC')}


In [16]:
# ---------------- PARÁMETROS DEL TRADE ---------------- #

horizon = 4  # Horizonte de tiempo para la reversión entrenada en el modelo de escaneo
entry_price = 0.0755712  # Precio de entrada REAL (ejecutado por ti)

#symbol = "UBUSDT"
#direction = "LONG"
#atr_15m = 0.000528   # ATR calculado en 15m en el momento de entrada calculado por la función "confirm_entry_15m_realtime" 

symbol = entry["Symbol"]
direction = entry["Direction"]
atr_15m = entry["ATR_15m"]



# Tiempo máximo de monitoreo (ej: 4 horas)
end_time_utc = datetime.now(timezone.utc) + timedelta(hours=horizon)

# ---------------- INICIAR MONITOREO ---------------- #

result = monitor_trade(
    symbol=symbol,
    direction=direction,
    entry_price=entry_price,
    atr_15m=atr_15m,
    horizon=horizon,
    end_time_utc=end_time_utc,
    poll_seconds=60  # revisar cada 1 minuto
)

print("Resultado final del trade:", result)


🟢 *TRADE OPEN*
         Symbol: `SPORTFUNUSDT`
         Direction: *SHORT*
         Entry: `0.075571`
         SL: `0.078813`
         TP Partial: `0.07314`
         TP: `0.068817`
⚠️ *SALIDA ESTRUCTURAL*
SPORTFUNUSDT SHORT
 Price: `0.07657`
Resultado final del trade: {'Status': 'STRUCTURAL_EXIT', 'Exit': 0.07657}
